In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.7 Serve via inference API

> 🎯 **Goal:** Put the trained model behind a clean request/response handler so the same logic can run in a test, a queue worker, or an HTTP route unchanged.

**Why this matters.** This is the last mile: turning a model object into something the outside world can call. Everything in this unit, the KV cache, batching, quantization, exists to make this handler fast and cheap. Getting the boundary right (a plain request in, a plain response out) is what lets you swap transports, test in-process, and keep your continuous integration green without spinning up a real server.

**The intuition.** Serving is just wrapping the model in a function that takes a request and returns a response. Think of a restaurant pass-through window: the kitchen (your model) does not care whether the order arrived by phone, walk-up, or app, it just receives an order and hands back a dish. Keep the model logic transport-agnostic and you can put any delivery mechanism in front of it later.

**The mechanics.** `InferenceService` holds the model and tokenizer and exposes one method, `handle`, that takes a request dict (a prompt, how many tokens to generate, a seed) and returns a response dict (the prompt echoed back plus the completion). That is the entire contract. Because `handle` is plain Python with no web framework inside it, the *same* method is what a unit test calls, what a background queue worker calls, and what an HTTP route calls. We exercise it directly below, then show (but do not run) the thin FastAPI wrapper that would expose it over HTTP.

In [ ]:
from pocketlm import train_tiny, pick_config, InferenceService

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
svc = InferenceService(model, tok)
resp = svc.handle({"prompt": "The ", "max_new_tokens": 20, "seed": 0})
print(resp["completion"])

**What just happened.** We built an `InferenceService`, called `handle` with a request dict, and printed the completion. No web server, no network, just a function call. That is the exact code path a real route would run.

Wrapping it in FastAPI is a few lines. This is shown, not run here (we deliberately avoid starting a server in continuous integration), so treat it as the glue code, not something this notebook executes:

```python
from fastapi import FastAPI
app = FastAPI()
svc = InferenceService(model, tok)

@app.post("/generate")
def generate_route(req: dict):
    return svc.handle(req)
# uvicorn serves `app`; the handler is exactly what we tested above.
```

The route is a one-line shim: it receives the JSON body and forwards it to the same `handle` we tested in-process. Because the handler is what carries the logic, the in-process test above already covers the real behavior, the FastAPI layer only adds transport.

**What just happened (the wrapper).** The decorator turns `generate_route` into an HTTP POST endpoint at `/generate`, but the model logic lives entirely in `svc.handle`. Swap FastAPI for a queue consumer or a gRPC stub and nothing about the model changes.

⚠️ **Common pitfalls**
- Putting model logic inside the route function. Keep it in `handle` so it stays testable without a server.
- Assuming this notebook starts a server. It does not, the FastAPI block is illustrative; only the in-process `handle` call runs.
- Trusting raw request dicts. A real endpoint validates and bounds inputs (for example, capping `max_new_tokens`) before handing them to the model.

🏋️ **Try it yourself.** Call `svc.handle` again with a different `prompt` and a larger `max_new_tokens`, and confirm the response dict still echoes your prompt and returns a completion of the expected length. For a real run, save this wrapper to a file and launch it yourself with `uvicorn module:app`, outside CI.

That is the whole course end to end: tokenize, build, train, modernize, adapt, and now serve a decoder-only language model you wrote yourself.

In [ ]:
assert resp["prompt"] == "The "
assert len(resp["completion"]) == len("The ") + 20